In [ ]:
import scanpy as sc
import spice
import pandas as pd  
import anndata as ad
import numpy as np
import matplotlib
matplotlib.rcParams["pdf.fonttype"] = 42   
matplotlib.rcParams["ps.fonttype"]  = 42  
import matplotlib.pyplot as plt


In [ ]:
#read in adata
adata = ad.read_h5ad("adata_example.h5ad")
adata

In [ ]:
spice.tl.assign_state_labels(
    adata,
    score_key="EMT_hallmarks_tumour", # column in adata.obs with the continuous score
    n_states=4, # 2 (median split) or 4 (quartile split)
    tumour_mask_key="cancer_epithelial_label", # only label tumour cells; None labels all cells
    label_key="state_label", # destination column in adata.obs
)

In [ ]:
spice.tl.build_graph(
    adata,
    spatial_key="spatial", # key in adata.obsm
    n_neighbors=12, # number of nearest neighbors
    celltype_key="celltype_minor", # cell-type column in adata.obs
    label_key="state_label", # labels from step 1
    score_key="state_label", # continuous score (stored as node attribute)
    n_blocks=4, # grid divisions for spatial block assignment
)

Example running using just cell type to predict cell plasticity

In [ ]:
spice.tl.cross_validate(
    adata,
    feature_mode="celltype", # "celltype", "intrinsic" or "combined"
    num_folds=5,
    num_epochs=500,
    hidden_dim1=16,
    hidden_dim2=32,
    dropout=0.5,
    learning_rate=0.01,
    class_weights=True, # inverse-frequency weighting
    inductive_split=True, # spatial block split (True) or random node split (False)
    return_last=True, # use last epoch's predictions
    verbose=True)

Example running intrinsic and extrinsic info to predict cell plasticity

In [ ]:
spice.tl.cross_validate(
    adata,
    feature_mode="combined", # "celltype", "intrinsic" or "combined"
    num_folds=5,
    num_epochs=500,
    hidden_dim1=16,
    hidden_dim2=32,
    dropout=0.5,
    learning_rate=0.01,
    class_weights=True, # inverse-frequency weighting
    inductive_split=True, # spatial block split (True) or random node split (False)
    return_last=True, # use last epoch's predictions
    verbose=True,
    pca_key="PCs",
    n_pcs=7
)

In [ ]:
spice.tl.evaluate(adata)

In [ ]:
spice.pl.auc_per_class(
    adata,
    state_map={0: "EPI", 1: "H-EPI", 2: "H-MES", 3: "MES"}
)

In [ ]:
spice.tl.explain_nodes(
    adata,
    n_explanations=50,
    ig_steps=25,   
    seed=0,               
)



In [ ]:
fig, axes = spice.pl.node_importance_signed(adata, state_map={0: "EPI", 1: "H-EPI", 2: "H-MES", 3: "MES"},save="spice_node_importance_direction_NO_PCs.pdf")
plt.show()


In [ ]:
fig, axes = spice.pl.node_importance(adata,state_map={0: "EPI", 1: "H-EPI", 2: "H-MES", 3: "MES"},save="spice_node_importance_no_direction_no_PCs.pdf")
plt.show()


In [ ]:
spice.tl.explain_edges(
    adata,
    n_explanations=50,
    fold_index=0
)

In [ ]:
edges=spice.pl.edge_network(
    adata,
    state_map={0: "EPI", 1: "H-EPI", 2: "H-MES", 3: "MES"},
    alpha=0.05,
    save="spice_edge_network.pdf",return_data=True
)

In [ ]:
fig, ax = spice.pl.edge_network(adata, state=0, state_map={0: "EPI"})


In [ ]:
spice.tl.run_baseline(adata, k=10)
spice.tl.compare_baselines(adata, reference="GNN")
spice.pl.baseline_significance(adata, save="spice_baseline_sig.pdf")
